# Import Models 

In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import NSGAIIISampler
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import sys
import time
 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import optuna
 

C:\Users\hp\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Build the python script for simulation running 

In [2]:
ECLRUN_EXE = r"C:\ecl\macros\eclrun.exe"

BASE_DIR = r"C:\Users\hp\Desktop\Eclipse 1 optimisation approach"  # we did the simulation for TPE & NSGA on the same DATA file 

DATA_FILE_NAME = "Egg_Model_ECL.DATA"
SCHEDULE_FILE_NAME = "SCHEDULE_NEW.INC"
RSM_PATH = r"C:\Users\hp\Desktop\Eclipse 1 optimisation approach\EGG_MODEL_ECL.RSM"

In [3]:
def modify_files(params):
#____________________________________________________________
    sched_path = os.path.join(BASE_DIR, SCHEDULE_FILE_NAME)

    inj_map = {
        "INJECT1": params["inj_rate1"],
        "INJECT2": params["inj_rate2"],
        "INJECT3": params["inj_rate3"],
        "INJECT4": params["inj_rate4"],
        "INJECT5": params["inj_rate5"],
        "INJECT6": params["inj_rate6"],
        "INJECT7": params["inj_rate7"],
        "INJECT8": params["inj_rate8"],
    }

    with open(sched_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()

        for well, rate in inj_map.items():
            if stripped.startswith(f"'{well}'") and "'RATE'" in stripped:
                parts = stripped.split()

                # Find RATE keyword index safely
                rate_idx = parts.index("'RATE'") + 1
                parts[rate_idx] = str(int(rate))

                line = " ".join(parts) + "\n"

        new_lines.append(line)

    with open(sched_path, "w") as f:
        f.writelines(new_lines)

#_____________________________________________________________
    data_path = os.path.join(BASE_DIR, DATA_FILE_NAME)

    prod_map = {
        "PROD1": params["BHP1"],
        "PROD2": params["BHP2"],
        "PROD3": params["BHP3"],
        "PROD4": params["BHP4"],
    }
    with open(data_path, "r") as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        stripped = line.strip()

        for well, bhp in prod_map.items():
            if stripped.startswith(f"'{well}'") and "'BHP'" in stripped:
                parts = stripped.split()
                parts[-1] = f"{int(bhp)}/"
                line = " ".join(parts) + "\n"

        new_lines.append(line)

    with open(data_path, "w") as f:
        f.writelines(new_lines)

In [4]:
def run_eclipse():
    cmd = [
        ECLRUN_EXE,
        "eclipse",
        DATA_FILE_NAME
    ]
    subprocess.run(cmd, cwd=BASE_DIR, check=True)
    
def extract_FOPT_FWPT_from_RSM(rsm_path):

    with open(rsm_path, "r") as f:
        lines = f.readlines()

    inside_summary = False
    fopt_idx = fwpt_idx = None

    for line in lines:

        if "SUMMARY OF RUN" in line:
            inside_summary = True
            fopt_idx = fwpt_idx = None
            continue

        if not inside_summary:
            continue

        tokens = line.split()
        if not tokens:
            continue

        # Detect column headers
        if fopt_idx is None and "FOPT" in tokens and "FWPT" in tokens:
            fopt_idx = tokens.index("FOPT")
            fwpt_idx = tokens.index("FWPT")
            continue

        if fopt_idx is None:
            continue

        # ✅ Only data rows have numeric TIME
        if not tokens[0][0].isdigit():
            continue
        if float(tokens[0]) == 3600.0:
            return float(tokens[fopt_idx]), float(tokens[fwpt_idx])

# Define the TPE objective function

In [5]:
# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
N_TRIALS = 300
N_STARTUP = 42          # random start-up trials; TPE takes over after this
STUDY_NAME = "egg_tpe_mo"
STORAGE = "sqlite:///article_wt_optimazation.db"   # saved so you can re-plot without re-running 8 h

PARAM_BOUNDS = {
    **{f"BHP{i}": (150, 390) for i in range(1, 5)},
    **{f"inj_rate{i}": (50, 400) for i in range(1, 9)},
}

C_RANDOM, C_TPE, C_PARETO = "#a3a29b", "#2b7bd6", "#ee6a33"
plt.rcParams.update({
    "axes.facecolor": "#fafaf8", "axes.edgecolor": "#bbbbbb",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e6", "font.size": 13,
})


# ------------------------------------------------------------------
# Objective: timings are stored on the trial itself (user_attrs)
# ------------------------------------------------------------------
def objective(trial):
    # 1. Sampling (random during start-up, TPE afterwards)
    t0 = time.perf_counter()
    params = {name: trial.suggest_int(name, lo, hi)
              for name, (lo, hi) in PARAM_BOUNDS.items()}
    trial.set_user_attr("t_sampling", time.perf_counter() - t0)

    # 2. Write Eclipse input files
    modify_files(params)

    # 3. Eclipse simulation (time recorded even if the run crashes)
    t0 = time.perf_counter()
    try:
        run_eclipse()
    finally:
        trial.set_user_attr("t_eclipse", time.perf_counter() - t0)

    # 4. Objectives
    fopt, fwpt = extract_FOPT_FWPT_from_RSM(RSM_PATH)
    return fopt, fwpt


# ------------------------------------------------------------------
# Figure 1: sampled values
# ------------------------------------------------------------------
def plot_sampled_values(study, n_startup, window=30, fname="sampled_values.png"):
    trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pareto = {t.number for t in study.best_trials}
    num = np.array([t.number for t in trials])
    is_rnd = num < n_startup
    is_par = np.isin(num, list(pareto))

    fig, axes = plt.subplots(3, 4, figsize=(24, 14), sharex=True)
    for ax, name in zip(axes.flat, PARAM_BOUNDS):
        val = np.array([t.params[name] for t in trials], dtype=float)

        # 10-90 % band of the last `window` TPE samples
        tn, tv = num[~is_rnd], val[~is_rnd]
        if len(tv) > 1:
            lo = [np.percentile(tv[max(0, i - window + 1):i + 1], 10) for i in range(len(tv))]
            hi = [np.percentile(tv[max(0, i - window + 1):i + 1], 90) for i in range(len(tv))]
            ax.fill_between(tn, lo, hi, color=C_TPE, alpha=0.12, lw=0, step="post")

        ax.scatter(num[is_rnd], val[is_rnd], s=18, c=C_RANDOM, zorder=2)
        ax.scatter(tn, tv, s=18, c=C_TPE, zorder=3)
        ax.scatter(num[is_par], val[is_par], s=90, facecolors="none",
                   edgecolors=C_PARETO, lw=2, zorder=4)

        b_lo, b_hi = PARAM_BOUNDS[name]
        ax.set_ylim(b_lo, b_hi)
        ax.set_title(name, fontsize=17)
    for ax in axes[-1]:
        ax.set_xlabel("Trial number", fontsize=15)

    handles = [
        Line2D([], [], ls="", marker="o", color=C_RANDOM, label="random start-up"),
        Line2D([], [], ls="", marker="o", color=C_TPE, label="TPE"),
        Line2D([], [], ls="", marker="o", mfc="none", mec=C_PARETO, mew=2, ms=10,
               label="Pareto-optimal"),
    ]
    fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False,
               fontsize=15, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(f"Sampled values: random start-up ({n_startup} trials), then concentration of "
                 f"TPE samples (band = 10-90 % of the last {window} TPE samples)",
                 y=0.965, fontsize=17)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(fname, dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------
# Figure 2: time per trial (Eclipse vs sampling) + totals
# ------------------------------------------------------------------
def plot_timing(study, n_startup, fname="time_per_trial.png"):
    trials = [t for t in study.trials
              if "t_eclipse" in t.user_attrs and "t_sampling" in t.user_attrs]
    num = np.array([t.number for t in trials])
    t_sim = np.array([t.user_attrs["t_eclipse"] for t in trials])
    t_smp = np.array([t.user_attrs["t_sampling"] for t in trials])
    is_rnd = num < n_startup

    fig = plt.figure(figsize=(22, 7.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.75, 1], wspace=0.04)

    # Left: per-trial times, log scale
    ax = fig.add_subplot(gs[0])
    ax.scatter(num, t_sim, s=22, c=C_TPE, label="Eclipse simulation")
    ax.scatter(num[is_rnd], t_smp[is_rnd], s=22, c=C_RANDOM, label="random start-up sampling")
    ax.scatter(num[~is_rnd], t_smp[~is_rnd], s=22, c=C_PARETO, label="TPE sampling")
    ax.set_yscale("log")
    ax.axvline(n_startup - 0.5, ls="--", color="#999999")
    ax.text(n_startup + 2, 1.0, "TPE starts", transform=ax.get_xaxis_transform(),
            va="bottom", color="#666666")
    ax.set_xlabel("Trial number")
    ax.set_ylabel("Time per trial (s, log scale)")
    ax.set_title("Time per trial: Eclipse vs TPE", pad=28)
    ax.legend(loc="center right", frameon=False)

    # Right: totals
    h_sim, h_smp = t_sim.sum() / 3600, t_smp.sum() / 3600
    tot = h_sim + h_smp
    ax2 = fig.add_subplot(gs[1])
    ax2.barh([0], [h_sim], color=C_TPE, height=0.5)
    ax2.barh([0], [h_smp], left=[h_sim], color=C_PARETO, height=0.5)
    ax2.set_ylim(-1.6, 0.5)
    ax2.set_yticks([])
    ax2.grid(axis="y", visible=False)
    ax2.set_xlabel("Total time (h)")
    ax2.set_title(f"Total {tot:.2f} h for {len(trials)} trials", pad=28)
    ax2.legend(handles=[
        Patch(color=C_TPE, label=f"Eclipse: {h_sim:.3f} h ({100 * h_sim / tot:.2f} %)"),
        Patch(color=C_PARETO, label=f"Sampling (random + TPE): {h_smp:.4f} h "
                                    f"({100 * h_smp / tot:.3f} %)"),
    ], loc="lower left", frameon=False, fontsize=14)

    fig.subplots_adjust(left=0.06, right=0.98, bottom=0.12, top=0.86)
    fig.savefig(fname, dpi=200)
    plt.close(fig)


# ------------------------------------------------------------------
if __name__ == "__main__":
    if "--plot-only" in sys.argv:
        study = optuna.load_study(study_name=STUDY_NAME, storage=STORAGE)
    else:
        sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=N_STARTUP)
        study = optuna.create_study(study_name=STUDY_NAME, storage=STORAGE,
                                    directions=["maximize", "minimize"],
                                    sampler=sampler, load_if_exists=True)
        study.set_user_attr("n_startup", N_STARTUP)
        # catch=...: a crashed Eclipse run is logged as FAIL instead of stopping the study
        study.optimize(objective, n_trials=N_TRIALS, catch=(Exception,))

    n_startup = study.user_attrs.get("n_startup", N_STARTUP)
    plot_sampled_values(study, n_startup)
    plot_timing(study, n_startup)
    print(f"Pareto-optimal trials: {len(study.best_trials)}")

[I 2026-11-12 20:26:22,927] A new study created in RDB with name: egg_tpe_mo
[I 2026-11-12 20:27:25,536] Trial 0 finished with values: [542456.4, 5351730.0] and parameters: {'BHP1': 240, 'BHP2': 379, 'BHP3': 326, 'BHP4': 294, 'inj_rate1': 104, 'inj_rate2': 104, 'inj_rate3': 70, 'inj_rate4': 354, 'inj_rate5': 260, 'inj_rate6': 298, 'inj_rate7': 57, 'inj_rate8': 390}. 
[I 2026-11-12 20:28:51,058] Trial 1 finished with values: [538358.4, 4629445.0] and parameters: {'BHP1': 350, 'BHP2': 201, 'BHP3': 193, 'BHP4': 194, 'inj_rate1': 156, 'inj_rate2': 234, 'inj_rate3': 201, 'inj_rate4': 152, 'inj_rate5': 264, 'inj_rate6': 98, 'inj_rate7': 152, 'inj_rate8': 178}. 
[I 2026-11-12 20:31:38,845] Trial 2 finished with values: [541525.1, 6195296.0] and parameters: {'BHP1': 259, 'BHP2': 339, 'BHP3': 198, 'BHP4': 273, 'inj_rate1': 257, 'inj_rate2': 66, 'inj_rate3': 263, 'inj_rate4': 109, 'inj_rate5': 72, 'inj_rate6': 383, 'inj_rate7': 388, 'inj_rate8': 333}. 
[I 2026-11-12 20:33:53,283] Trial 3 finishe

Pareto-optimal trials: 34
